In [0]:
%pip install vaderSentiment
%restart_python

In [0]:
import requests
from pathlib import Path
from pyspark.sql.functions import col
from pyspark.sql import Row
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import html
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from time import time as epoch_time
import mimetypes


In [0]:
def save_post(post_id: str, content: str, media_urls: list[str], schema: str):
    volume_base = f"/Volumes/3PO/{schema}/{post_id}"
    spark.sql("CREATE SCHEMA IF NOT EXISTS `3PO`.`{schema}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `3PO`.`{schema}`.`{post_id}`")
    dbutils.fs.mkdirs(volume_base)

    # Save text content
    with open(f"{volume_base}/0.txt", "w", encoding="utf-8") as f:
        f.write(content)

    saved=1
    # Save media files
    for i, url in enumerate(media_urls):
        try:
            r = requests.get(url, timeout=15)
            r.raise_for_status()
            ext = get_extension(url, r)
            filepath = f"{volume_base}/{i + 1}.{ext}"
            with open(filepath, "wb") as f:
                f.write(r.content)
            saved += 1
        except requests.RequestException as e:
            print(f"  Warning: failed to download {url}: {e}")
    return saved

def get_extension(url: str, response: requests.Response) -> str:
    # 1. Try the @format suffix (e.g. ...@jpeg, ...@png)
    clean = url.split("?")[0]
    if "@" in clean:
        candidate = clean.rsplit("@", 1)[-1]
        if re.fullmatch(r"[a-zA-Z0-9]{1,10}", candidate):
            return candidate

    # 2. Try the last path segment's extension (e.g. .../photo.jpg)
    last_segment = clean.rstrip("/").rsplit("/", 1)[-1]
    if "." in last_segment:
        candidate = last_segment.rsplit(".", 1)[-1]
        if re.fullmatch(r"[a-zA-Z0-9]{1,10}", candidate):
            return candidate

    # 3. Fall back to Content-Type header
    content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
    if content_type:
        ext = mimetypes.guess_extension(content_type)
        if ext:
            return ext.lstrip(".")

    # 4. Last resort
    return "bin"

In [0]:
def get_trending_mastodon(time: int = 0, instance: str = "mastodon.social", limit: int = 10):

    #ensure locations
    spark.sql("CREATE TABLE IF NOT EXISTS 3po.main.mastodon_positivity (id STRING, positivity FLOAT) USING DELTA")
    spark.sql("CREATE SCHEMA IF NOT EXISTS 3po.tmastodon")
    spark.sql("CREATE VOLUME IF NOT EXISTS 3po.responses.mastodon")

    #ingestion statistics variables
    p=0
    f=0

    #get data and tools
    url      = f"https://{instance}/api/v1/trends/statuses?limit={limit}"
    headers  = {"User-Agent": "trending_sentiment_script/1.0"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    posts    = response.json()
    analyzer = SentimentIntensityAnalyzer()
    path = f"/Volumes/3po/responses/mastodon/{time}.json"
    dbutils.fs.put(path, response.text, overwrite=True)
   
    #make timestamp table
    table_name = f"3po.tmastodon.t{time}"
    sql = f"""CREATE TABLE IF NOT EXISTS {table_name} (id STRING,impressions INT,positivity FLOAT)USING DELTA"""
    spark.sql(sql)

    #initiate weighted average
    big_pos=0
    big_imp=0

    for post in posts:

        #approximate impressions
        impressions=(post.get("reblogs_count", 0)*10+post.get("favourites_count",0)+post.get("replies_count",0))

        #check if post reoccuring
        id=post["id"]
        row=(spark.table("3po.main.mastodon_positivity").filter(col("id")==id).select("positivity").first())

        #history name
        history_name=f"3po.hmastodon.h{id}"

        #handle reoccuring post
        if (row):
            positivity=row["positivity"]

        #handle new post
        else:
            p+=1
            #analyze textcontent
            text = post.get("content", "")
            text = re.sub(r'<a\b[^>]*>.*?</a>', ' ', text, flags=re.DOTALL)
            text = re.sub(r'<[^>]+>', ' ', text)
            text = html.unescape(text)
            text = re.sub(r'\s+', ' ', text).strip()
            positivity=analyzer.polarity_scores(text)["compound"] if text.strip() else None

            #Insert positivity
            schema = StructType([StructField("id", StringType(), True),StructField("positivity", FloatType(), True),])
            row_df_pos = spark.createDataFrame([Row(id=id, positivity=positivity)],schema=schema)
            row_df_pos.select("id", "positivity").write.insertInto("3po.main.mastodon_positivity")

            #make history
            sql = f"""CREATE TABLE IF NOT EXISTS {history_name} (time LONG,impressions INT)USING DELTA"""
            spark.sql(sql)

            # Save post
            media_urls = [a["url"] for a in post.get("media_attachments", []) if a.get("url")]
            f+=save_post(id, text, media_urls, "cmastodon")

        #insert into timestamp
        schema = StructType([StructField("id", StringType(), True),StructField("impressions", IntegerType(), True),StructField("positivity",FloatType(), True),])
        row_df = spark.createDataFrame([Row(id=id, impressions=impressions, positivity=positivity)], schema=schema)
        row_df.write.insertInto(table_name)
        
        #insert into history
        row_df_hist = spark.createDataFrame([Row(time=time, impressions=impressions)])
        row_df_hist.select("time", "impressions").write.insertInto(history_name)

        #weighted average
        if positivity:
            big_pos+=positivity*impressions
            big_imp+=impressions
    if big_imp>0:
        return [big_pos/big_imp, p, f]
    else:
        return [None, p, f]

    


In [0]:
def get_trending_bluesky(time: int = 0, limit: int = 10):

    #ensure locations
    spark.sql("CREATE TABLE IF NOT EXISTS 3po.main.bluesky_positivity (id STRING, positivity FLOAT) USING DELTA")
    spark.sql("CREATE SCHEMA IF NOT EXISTS 3po.tbluesky")
    spark.sql("CREATE VOLUME IF NOT EXISTS 3po.responses.bluesky")

    #ingestion statistics variables
    p=0
    f=0

    #get data and tools
    BASE_URL = "https://public.api.bsky.app/xrpc"
    WHATS_HOT_URI = "at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.feed.generator/whats-hot"
    headers  = {"User-Agent": "trending_sentiment_script/1.0"}
    url      = f"{BASE_URL}/app.bsky.feed.getFeed"
    params   = {"feed": WHATS_HOT_URI, "limit": limit}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    items    = response.json().get("feed", [])
    analyzer = SentimentIntensityAnalyzer()
    path = f"/Volumes/3po/responses/bluesky/{time}.json"
    dbutils.fs.put(path, response.text, overwrite=True)
   
    #make timestamp table
    table_name = f"3po.tbluesky.t{time}"
    sql = f"""CREATE TABLE IF NOT EXISTS {table_name} (id STRING,impressions INT,positivity FLOAT)USING DELTA"""
    spark.sql(sql)

    #initiate weighted average
    big_pos=0
    big_imp=0

    for item in items:
        post   = item.get("post", {})

        #approximate impressions
        impressions=(post.get("likeCount",0)+post.get("repostCount",0)*10 +post.get("replyCount",0))

        #check if post reoccuring
        uri=post.get("uri", "")
        id=uri.split("/")[-1] if uri else post.get("cid", "unknown")
        row=(spark.table("3po.main.bluesky_positivity").filter(col("id")==id).select("positivity").first())

        #history name
        history_name=f"3po.hbluesky.h{id}"

        #handle reoccuring post
        if (row):
            positivity=row["positivity"]

        #handle new post
        else:
            p+=1
            #analyze textcontent
            record = post.get("record", {})
            text   = record.get("text", "")
            positivity=analyzer.polarity_scores(text)["compound"] if text.strip() else None

            #Insert positivity
            schema = StructType([StructField("id", StringType(), True),StructField("positivity", FloatType(), True),])
            row_df_pos = spark.createDataFrame([Row(id=id, positivity=positivity)],schema=schema)
            row_df_pos.select("id", "positivity").write.insertInto("3po.main.bluesky_positivity")

            #make history
            sql = f"""CREATE TABLE IF NOT EXISTS {history_name} (time LONG,impressions INT)USING DELTA"""
            spark.sql(sql)

            # Save post
            media_urls=[]
            embed=post.get("embed",{})
            embed_type=embed.get("$type","")
            if embed_type=="app.bsky.embed.images#view":
                for img in embed.get("images", []):
                    url = bluesky_img_url(img["fullsize"])
                    if url:
                        media_urls.append(url)
            elif embed_type == "app.bsky.embed.recordWithMedia#view":
                for img in embed.get("media", {}).get("images", []):
                    url = bluesky_img_url(img["fullsize"])
                    if url:
                        media_urls.append(url)
            f+=save_post(id, text, media_urls, "cbluesky")

        #insert into timestamp
        schema = StructType([StructField("id", StringType(), True),StructField("impressions", IntegerType(), True),StructField("positivity",FloatType(), True),])
        row_df = spark.createDataFrame([Row(id=id, impressions=impressions, positivity=positivity)], schema=schema)
        row_df.write.insertInto(table_name)
        
        #insert into history
        row_df_hist = spark.createDataFrame([Row(time=time, impressions=impressions)])
        row_df_hist.select("time", "impressions").write.insertInto(history_name)

        #weighted average
        if positivity:
            big_pos+=positivity*impressions
            big_imp+=impressions
    if big_imp>0:
        return [big_pos/big_imp, p, f]
    else:
        return [None, p, f]


def bluesky_img_url(fullsize: str) -> str | None:
    # Already a real URL
    if fullsize.startswith("http"):
        return fullsize
    # at://did:.../cid@format  or  .../plain/did/cid@format
    try:
        # Strip leading slashes/path, split on known separators
        parts = fullsize.replace("@jpeg", "").replace("@png", "").replace("@webp", "").split("/")
        did   = next(p for p in parts if p.startswith("did:"))
        cid   = parts[-1]
        return f"https://cdn.bsky.app/img/feed_fullsize/plain/{did}/{cid}@jpeg"
    except (StopIteration, IndexError):
        print(f"  Warning: could not parse image ref: {fullsize}")
        return None


In [0]:
def sample(time=0, limit=10):
    PLATFORM_COUNT=2 #UPDATE THIS

    schema = StructType([StructField("time", IntegerType(), True),
    StructField("bluesky_positivity", FloatType(), True),
    StructField("mastodon_positivity", FloatType(), True),
    #MORE GOES HERE
    ])

    mastodon=get_trending_mastodon(time,"mastodon.social", limit)
    bluesky=get_trending_bluesky(time, limit)

    row_df = spark.createDataFrame([Row(time=time, 
        bluesky_positivity=bluesky[0],
        mastodon_positivity=mastodon[0],
        #MORE GOES HERE
        )],schema=schema)                           
    row_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("3po.main.timeline")


    schema = StructType([StructField("time", IntegerType(), True),StructField("instances", IntegerType(), True),
    StructField("posts",IntegerType(), True),StructField("files", IntegerType(), True)])

    row_df = spark.createDataFrame([Row(time=time, instances=limit*PLATFORM_COUNT,
        posts=bluesky[1]+mastodon[1]#MORE GOES HERE
        ,files=bluesky[2]+mastodon[2]#MORE GOES HERE
        ,)],schema=schema)
    row_df.write.format("delta").mode("append").saveAsTable("3po.main.ingests")    

def ingest(samples=1, size=10, period=86400 ):
    i=0
    time=(int)(epoch_time())
    while (i!=samples):
        if epoch_time()>=time:
            sample(time, size)
            i+=1
            time=time+period

    

In [0]:
time=(int)(epoch_time())
sample(time,10)
